# NILM – Semaine 2 : un premier modèle et sa précision

**Objectif de la semaine.** Séparer la courbe de la maison entière en appareils avec un algorithme **simple, sans apprentissage profond**, puis **mesurer sa précision** sur une période qu'il n'a jamais vue. Ce premier modèle me sert de référence (*baseline*) : le modèle d'IA des semaines suivantes ne sera gardé que s'il fait mieux.

Je compare deux méthodes :
1. **Naïf** : « l'appareil est toujours allumé ». C'est le plancher : un modèle qui ne fait pas mieux ne sert à rien.
2. **Optimisation combinatoire (CO)** : j'apprends les niveaux de puissance de chaque appareil sur la période d'entraînement, puis, à chaque minute, je cherche la combinaison d'états qui colle le mieux à la puissance totale. C'est la méthode de base de NILMTK, que j'ai réécrite en quelques lignes avec `pandas` et `numpy`.

**Données.** UK-DALE, maison 1 : les fichiers `labels.dat` et `channel_N.dat` lus sur la fenêtre de mars à mai 2013 (le notebook a été exécuté sous Colab, avec les fichiers dans `/content`). Après suppression des minutes où un canal manque, il reste environ un mois exploitable (mars-avril 2013).

## 1. Réglages
- `MAINS` et `APPLIANCES` : le canal de la maison entière et les 4 appareils étudiés (frigo, bouilloire, lave-linge, lave-vaisselle) ;
- `LOAD` : la fenêtre de dates lue dans les fichiers ;
- `TRAIN`, `TEST` : découpe automatique, **70 % du début pour l'entraînement et 30 % de la fin pour le test** (on peut aussi imposer des dates) ;
- `ON_W` : seuil au-dessus duquel un appareil est considéré allumé (30 W pour le frigo, 500 W pour la bouilloire, 20 W pour les lave-linge et lave-vaisselle) ;
- `N_ON_STATES` : nombre de niveaux de puissance appris par appareil (1 pour le frigo et la bouilloire, 3 pour les appareils à cycles).

In [ ]:
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# ================== RÉGLAGES (à adapter) ==================
DATA_DIR = Path("/content")
MAINS = "aggregate"                                             # nom du canal « maison entière »
APPLIANCES = ["fridge", "kettle", "washing_machine", "dishwasher"]
LOAD = ("2013-03-01", "2013-05-31")                             # fenêtre de dates lue dans les fichiers
TRAIN, TEST = None, None   # None = découpe automatique : 70 % du début pour l'entraînement, 30 % de la fin pour le test
# Pour imposer des dates : TRAIN = ("2013-03-01", "2013-03-21"); TEST = ("2013-03-22", "2013-04-02")
RESAMPLE = "1min"                                               # pas de temps
ON_W = {"fridge": 30, "kettle": 500, "washing_machine": 20, "dishwasher": 20}   # seuil « allumé » (W)
N_ON_STATES = {"fridge": 1, "kettle": 1, "washing_machine": 3, "dishwasher": 3} # niveaux de puissance appris par appareil
# ===========================================================
plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True

## 2. Chargement et découpe entraînement / test
Je lis chaque canal, je le ramène à la minute et je ne garde que les minutes où tous les canaux ont une mesure. La découpe est chronologique : le modèle apprend sur le début de la période et il est jugé sur la fin, qu'il n'a jamais vue. C'est la seule façon honnête de mesurer ce qu'il ferait sur de nouvelles données.

In [ ]:
def read_labels(folder):
    labels = {}
    for line in (folder / "labels.dat").read_text().splitlines():
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            labels[parts[1].strip()] = int(parts[0])
    return labels

def read_channel(folder, ch):
    df = pd.read_csv(folder / f"channel_{ch}.dat", sep=r"\s+", header=None, usecols=[0, 1], names=["t", "w"])
    df.index = pd.to_datetime(df["t"], unit="s")
    s = df["w"].astype(float)
    s = s[~s.index.duplicated()].sort_index()
    return s.loc[LOAD[0]:LOAD[1]]

labels = read_labels(DATA_DIR)
wanted = [MAINS] + APPLIANCES
series, used = {}, []
for name in wanted:
    if name not in labels:
        print("absent de labels.dat :", name); continue
    try:
        s = read_channel(DATA_DIR, labels[name])
    except FileNotFoundError:
        print(f"fichier manquant : channel_{labels[name]}.dat ({name})"); continue
    if s.empty:
        print("aucune donnée sur la période pour", name); continue
    series[name] = s.resample(RESAMPLE).mean()
    print(f"chargé : {name:16s} {len(s):>9} mesures  {s.index.min()} -> {s.index.max()}")
if MAINS not in series:
    raise SystemExit("Canal maison entière introuvable : vérifie les fichiers envoyés dans /content.")
APPLIANCES = [a for a in APPLIANCES if a in series]
data = pd.DataFrame(series)
n0 = len(data)
data = data.dropna()                     # on retire les minutes où un canal a un trou
print(f"\nminutes conservées : {len(data)} / {n0} ({100*(1-len(data)/n0):.1f} % retirées pour cause de trous)")
if TRAIN is None or TEST is None:
    cut = data.index[int(len(data) * 0.7)]
    train, test = data.loc[:cut], data.loc[cut:].iloc[1:]
else:
    train, test = data.loc[TRAIN[0]:TRAIN[1]], data.loc[TEST[0]:TEST[1]]
print("entraînement :", len(train), "pas |", train.index.min(), "->", train.index.max())
print("test         :", len(test), "pas |", test.index.min(), "->", test.index.max())
if train.empty or test.empty:
    raise SystemExit("Entraînement ou test vide : vérifie LOAD, TRAIN / TEST et les fichiers envoyés.")

## 3. Ce que le modèle ne pourra pas expliquer
J'estime la consommation « de fond » (éclairage, TV, veilles, appareils non suivis) comme la médiane, sur l'entraînement, de (total − somme des appareils suivis). Je la retire du signal avant de chercher les appareils. J'affiche aussi la part de l'énergie de la maison qui vient des 4 appareils suivis.

Sur mes données, le fond est estimé à environ **201 W** et les 4 appareils suivis représentent environ un quart de l'énergie.

In [ ]:
other = (train[MAINS] - train[APPLIANCES].sum(axis=1)).clip(lower=0)
OFFSET = float(other.median())
print(f"Consommation de fond estimée : {OFFSET:.0f} W")
part = train[APPLIANCES].sum().sum() / train[MAINS].sum() * 100
print(f"Part de l'énergie de la maison venant des {len(APPLIANCES)} appareils suivis : {part:.0f} %")

## 4. Apprentissage des niveaux de puissance
Pour chaque appareil, je regroupe ses puissances « allumé » de l'entraînement en quelques niveaux, avec un k-means à une dimension écrit à la main. Un frigo a un seul niveau ; un lave-linge en a plusieurs (chauffe de l'eau, rotation, essorage). Chaque appareil a donc l'état 0 (éteint) plus ses niveaux appris.

Niveaux obtenus (W) : frigo 90 ; bouilloire 1 735 ; lave-linge 161 / 1 010 / 1 866 ; lave-vaisselle 119 / 1 147 / 2 325.

In [ ]:
def kmeans1d(x, k, iters=100):
    if len(x) > 50000:
        x = np.random.default_rng(0).choice(x, 50000, replace=False)
    c = np.quantile(x, np.linspace(0.1, 0.9, k)) if k > 1 else np.array([x.mean()])
    for _ in range(iters):
        lab = np.abs(x[:, None] - c[None, :]).argmin(1)
        new = np.array([x[lab == j].mean() if (lab == j).any() else c[j] for j in range(k)])
        if np.allclose(new, c):
            break
        c = new
    return np.sort(c)

states = {}
for a in APPLIANCES:
    on = train[a][train[a] > ON_W[a]].values
    if len(on) < 10:
        print(f"{a}: presque jamais allumé sur l'entraînement -> ignoré"); states[a] = np.array([0.0]); continue
    k = min(N_ON_STATES[a], max(1, len(np.unique(on.round()))))
    states[a] = np.concatenate([[0.0], kmeans1d(on, k)])
    print(f"{a:16s} niveaux appris (W) : {np.round(states[a], 0)}")

## 5. Désagrégation sur la période de test
**CO** : je construis toutes les combinaisons d'états (un état par appareil), soit 2 × 2 × 4 × 4 = **64 combinaisons**. À chaque minute du test, je garde celle dont la somme est la plus proche de la puissance totale moins la consommation de fond. Le calcul est vectorisé par paquets de 20 000 minutes.

**Naïf** : chaque appareil est déclaré allumé en permanence, à sa puissance moyenne de fonctionnement.

In [ ]:
combo_states = np.array(list(product(*[states[a] for a in APPLIANCES])))   # (nb_combinaisons, nb_appareils)
combo_sums = combo_states.sum(axis=1)
print(len(combo_sums), "combinaisons d'états testées à chaque pas de temps")

y = (test[MAINS] - OFFSET).clip(lower=0).values
idx = np.empty(len(y), dtype=int)
for i in range(0, len(y), 20000):
    idx[i:i + 20000] = np.abs(y[i:i + 20000, None] - combo_sums[None, :]).argmin(1)
co_pred = pd.DataFrame(combo_states[idx], index=test.index, columns=APPLIANCES)

naive_pred = pd.DataFrame({a: np.where(True, train[a][train[a] > ON_W[a]].mean() if (train[a] > ON_W[a]).any() else 0.0, 0.0)
                           for a in APPLIANCES}, index=test.index)        # « toujours allumé »
print("terminé")

## 6. Précision : le tableau à retenir
- **F1** (0 à 1) : l'appareil est-il détecté allumé au bon moment ? 1 = parfait. Il combine la **précision** (quand on dit « allumé », a-t-on raison ?) et le **rappel** (trouve-t-on tous les moments où il est allumé ?) ;
- **MAE (W)** : écart moyen entre puissance prédite et réelle, à chaque minute ;
- **erreur d'énergie** : écart relatif sur les kWh attribués à l'appareil sur la période de test. C'est la métrique qui compte pour la facture et pour la flexibilité.

Le graphique compare le F1 du naïf et du CO pour chaque appareil.

In [ ]:
dt_h = pd.Timedelta(RESAMPLE).total_seconds() / 3600

def metrics(y_true, y_pred, thr):
    t, p = y_true > thr, y_pred > thr
    tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
    e_t, e_p = y_true.sum() * dt_h / 1000, y_pred.sum() * dt_h / 1000
    return {
        "F1": 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan,
        "précision": tp / (tp + fp) if (tp + fp) else np.nan,
        "rappel": tp / (tp + fn) if (tp + fn) else np.nan,
        "MAE (W)": float(np.abs(y_true - y_pred).mean()),
        "kWh réels": e_t, "kWh prédits": e_p,
        "erreur énergie (%)": 100 * abs(e_p - e_t) / e_t if e_t > 0 else np.nan,
    }

rows = []
for method, pred in [("Naïf (toujours allumé)", naive_pred), ("CO", co_pred)]:
    for a in APPLIANCES:
        rows.append({"méthode": method, "appareil": a, **metrics(test[a].values, pred[a].values, ON_W[a])})
results = pd.DataFrame(rows).set_index(["appareil", "méthode"]).sort_index()
results.round(2)

In [ ]:
f1 = results["F1"].unstack("méthode")
ax = f1.plot.bar(rot=0, color=["#B0BEC5", "#0E7C86"], title="F1 par appareil (période de test)", ylim=(0, 1))
ax.set_ylabel("F1 (1 = parfait)"); plt.show()

## 7. Une journée de test, appareil par appareil (réel vs prédit)
J'affiche la journée de test la plus active et, pour chaque appareil, la puissance réelle et celle prédite par le CO. Ce graphique montre concrètement les erreurs du CO : plusieurs appareils activés en même temps pour un seul pic, et des détections hachées.

In [ ]:
day_energy = test[APPLIANCES].sum(axis=1).groupby(test.index.date).sum()
day = str(day_energy.idxmax())                      # jour de test avec le plus d'activité
print("jour affiché :", day)
fig, axes = plt.subplots(len(APPLIANCES), 1, figsize=(11, 2.2 * len(APPLIANCES)), sharex=True)
for ax, a in zip(np.atleast_1d(axes), APPLIANCES):
    ax.plot(test[a].loc[day], color="#D99200", label="réel")
    ax.plot(co_pred[a].loc[day], color="#0E7C86", label="CO", alpha=0.85)
    ax.set_title(a + " (W)", fontsize=10); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

## 8. Résultats et analyse
| Appareil | kWh réels | kWh prédits (CO) | F1 CO | F1 naïf | Précision CO | Rappel CO | Erreur d'énergie CO |
|---|---|---|---|---|---|---|---|
| Lave-linge | 12,19 | 22,18 | 0,34 | 0,11 | 0,22 | 0,82 | 82 % |
| Bouilloire | 6,91 | 9,89 | 0,25 | 0,02 | 0,21 | 0,32 | 43 % |
| Lave-vaisselle | 7,62 | 27,84 | 0,11 | 0,05 | 0,06 | 0,65 | 265 % |
| Frigo | 15,25 | 10,80 | 0,48 | 0,59 | 0,57 | 0,41 | 29 % |

Le CO bat le naïf sur 3 appareils sur 4, mais **aucun F1 ne dépasse 0,5**, et le frigo est même moins bien détecté que par « toujours allumé ». Le lave-linge est bien retrouvé (rappel 0,82) mais 4 détections sur 5 sont fausses ; le CO attribue au lave-vaisselle plus de trois fois son énergie réelle. Sur la journée affichée, il active plusieurs fois en même temps le lave-linge, le lave-vaisselle et la bouilloire alors qu'un seul de ces appareils tourne.

**Causes identifiées :**
1. la bouilloire (2,4 kW) et les chauffes du lave-linge (≈ 2 kW) et du lave-vaisselle (2,3 kW) ont presque la même puissance : le CO ne peut pas les distinguer sur une seule valeur ;
2. environ 75 % de l'énergie vient d'appareils non suivis, dont les pics sont attribués à tort aux appareils connus ;
3. le frigo (90 W) est plus faible que les variations du fond (≈ 200 W) : sa détection clignote ;
4. la moyenne à la minute écrase les impulsions de 2-3 minutes de la bouilloire (niveau appris à 1 735 W au lieu d'environ 2,4 kW) ;
5. le total est en VA et les appareils en W, et le fond est supposé constant ;
6. la décision est prise minute par minute, sans mémoire : les prédictions sont hachées au lieu de former des cycles continus.

Les causes 1 et 6 tiennent à la méthode elle-même : elle ne regarde que la puissance de l'instant, jamais la forme ni la durée. C'est ce qu'un réseau de neurones peut corriger.

## 9. Conclusion
- **Modèle** : optimisation combinatoire (CO) sur 4 appareils, avec 1 à 3 niveaux de puissance par appareil appris par k-means, soit 64 combinaisons testées à chaque minute. Aucun apprentissage profond, quelques lignes de `numpy`.
- **Entrée** : la puissance totale de la maison à la minute (moins la consommation de fond estimée).
- **Sorties** : pour chaque minute, l'état et la puissance de chaque appareil (le niveau choisi).
- **Performances** : F1 de 0,11 à 0,48 selon l'appareil, erreur d'énergie de 29 à 265 %, et un frigo moins bien détecté que par le modèle naïf. Ce niveau est insuffisant pour piloter la flexibilité ou afficher une consommation fiable par appareil.
- **Intérêt** : ce baseline fixe la barre à dépasser et explique *pourquoi* il faut un modèle d'IA : il faut regarder la **forme de la courbe sur plusieurs heures** pour distinguer des appareils de même puissance et produire des cycles continus.
- **Décision pour la suite** : en semaine 3, j'entraîne un petit réseau de neurones convolutif sur une année complète de données, comparé au CO **sur le même test**. Règle fixée à l'avance : le réseau n'est gardé que s'il bat le CO.